In [ ]:
from js import location
import micropip

_path = location.pathname
_root = _path.split("/extensions/")[0] if "/extensions/" in _path else _path.rsplit("/", 1)[0]
_pypi = location.origin + _root + "/pypi/"
_wheels = [
    'lark-1.3.1-py3-none-any.whl',
    'icontract-2.7.3-py3-none-any.whl',
    'asttokens-2.4.1-py2.py3-none-any.whl',
    'typing_extensions-4.16.0-py3-none-any.whl',
    'six-1.17.0-py2.py3-none-any.whl',
    'comm-0.2.3-py3-none-any.whl',
    'fpl_walker-0.0.0-py3-none-any.whl',
]
await micropip.install([_pypi + w for w in _wheels], deps=False)
import fpl
fpl


In [ ]:
import marimo as mo

# 01.01.05 Codec

Every representation a pass reads or writes gets a codec: `fonify` writes the value as [FON](01.01.02-fon.md), `defonify` reads it back unchanged ([ADR-0005](../../02-docs/02.05-project/adr/ADR-0005-fonify-defonify.md)). The trip has two levels. `to_fon` materialises an object as a FON value and `from_fon` builds it back; `fon.write` and `fon.read` carry that value to bytes and back. Only the first level is written here: the text FON takes is the FON module's business, so a codec never spells it.

The materialisation is one generic reflection over the frozen dataclasses that hold the program: the surface tree, the core tree, the lexer's tokens and the span. Nothing here knows a pass, so reading FON never needs the compiler ([ADR-0004](../../02-docs/02.05-project/adr/ADR-0004-native-abts-fon-at-boundaries.md)).

In [ ]:
"""
The codecs, keyed by the name of the representation they carry.
"""

import dataclasses
from dataclasses import dataclass
from decimal import Decimal
from typing import cast

from fpl import ast_core, ast_surface, errors
from fpl.errors import FplError, Span
from fpl.fon import (
    Absent,
    Bool,
    Dict,
    List,
    Num,
    Str,
    Sym,
    Tagged,
    Value,
    read,
    write,
)
from fpl.lex import Prelexed, Stashed, Tokens


def _declared(module: object) -> list[type]:
    """
    The dataclasses a module defines, not those it imports.
    """
    return [
        kind
        for kind in vars(module).values()
        if isinstance(kind, type)
        and dataclasses.is_dataclass(kind)
        and kind.__module__ == getattr(module, "__name__", None)
    ]


CLASSES: dict[str, type] = {
    kind.__name__: kind
    for kind in (
        *_declared(ast_surface),
        *_declared(ast_core),
        *_declared(errors),
        Stashed,
        Prelexed,
        Tokens,
    )
}
"""
Every class a value may be built of, by the name its FON tag gives.
"""
if len(CLASSES) != len(
    (
        *_declared(ast_surface),
        *_declared(ast_core),
        *_declared(errors),
        Stashed,
        Prelexed,
        Tokens,
    )
) or "Decimal" in CLASSES:
    raise ImportError("two FON tags would name one class")

## Object to FON value

A dataclass is a tagged tuple, its class name first and then every field in declaration order, spans and keyword-only fields too: a span takes no part in equality, so a codec that dropped it would still compare equal and lose the position ([ADR-0009](../../02-docs/02.05-project/adr/ADR-0009-keep-information-strip-at-release.md)). A tuple is a list, a dict a FON dict. The scalars keep their kinds apart: `True` is a FON boolean, never the number 1; an `int` is a bare number; a `Decimal` is tagged and carries its exact text, so `2`, `2.0` and `2.50` are three values; text is a string, while a core `Symbol` is a dataclass and so tagged; `None` is absence. An error is tagged by its class, with its span and message.

In [ ]:
def to_fon(obj: object) -> Value:
    """
    The FON value of an object of the program's representations; any
    other object is refused with a TypeError.
    """
    match obj:
        case None:
            return Absent()
        case bool():
            return Bool(obj)
        case int():
            return Num(Decimal(obj))
        case Decimal():
            return Tagged((Sym("Decimal"), Str(str(obj))))
        case str():
            return Str(obj)
        case tuple():
            items = cast(tuple[object, ...], obj)
            return List(tuple(map(to_fon, items)))
        case dict():
            entries = cast(dict[object, object], obj)
            return Dict(
                tuple((_key(k), to_fon(v)) for k, v in entries.items())
            )
        case FplError():
            return Tagged(
                (Sym(type(obj).__name__), to_fon(obj.span), Str(obj.message))
            )
        case _ if CLASSES.get(type(obj).__name__) is type(obj):
            return Tagged(
                (
                    Sym(type(obj).__name__),
                    *(
                        to_fon(getattr(obj, field.name))
                        for field in dataclasses.fields(cast(Span, obj))
                    ),
                )
            )
        case _:
            raise TypeError(f"no FON for a {type(obj).__name__}")


def _key(obj: object) -> Num | Str:
    """
    A dict key: a number or a text, the leaves a FON key may be.
    """
    key = to_fon(obj)
    if not isinstance(key, Num | Str):
        raise TypeError(f"no FON key for a {type(obj).__name__}")
    return key

## FON value to object

`from_fon` is the inverse, and as strict as the FON reader: a number with a point where an `int` belongs, a tag no class answers to, or a field count other than the class's is refused with a ValueError rather than guessed at. A dataclass is built with keywords, so a keyword-only field lands where it was. An error is built without its class's own constructor, which may take fewer arguments (`UnderflowError` takes only a span), so every subclass of `FplError` comes back as itself.

In [ ]:
def from_fon(value: Value) -> object:
    """
    The object a FON value materialises, the inverse of to_fon.
    """
    match value:
        case Absent():
            return None
        case Bool():
            return value.value
        case Num() if value.value == value.value.to_integral_value():
            return int(value.value)
        case Str():
            return value.text
        case List():
            return tuple(map(from_fon, value.items))
        case Dict():
            return {from_fon(k): from_fon(v) for k, v in value.entries}
        case Tagged(items=(Sym(name=name), *rest)):
            return _built(name, tuple(rest))
        case _:
            raise ValueError(f"no object for FON {write(value)}")


def _built(name: str, rest: tuple[Value, ...]) -> object:
    """
    The object a tag and its items name.
    """
    match name, rest:
        case "Decimal", (Str(text=text),):
            return Decimal(text)
        case _ if name in CLASSES:
            kind = CLASSES[name]
            fields = dataclasses.fields(kind)
            if len(fields) != len(rest):
                raise ValueError(f"{name} takes {len(fields)} fields")
            return kind(
                **{f.name: from_fon(v) for f, v in zip(fields, rest)}
            )
        case _, (span, Str(text=message)) if name in _errors():
            built = _errors()[name]
            error = built.__new__(built)
            FplError.__init__(error, cast(Span, from_fon(span)), message)
            return error
        case _:
            raise ValueError(f"no object for the FON tag {name}")


def _errors() -> dict[str, type[FplError]]:
    """
    FplError and every subclass loaded now, by name.
    """
    found: dict[str, type[FplError]] = {}
    todo = [FplError]
    while todo:
        kind = todo.pop()
        found[kind.__name__] = kind
        todo.extend(kind.__subclasses__())
    return found

## The document

A document on the pipe says what it carries: one tagged tuple whose tag is the representation's name and whose single item is the value. `fon.write` chooses how that is spelt. Decoding a document of another representation is refused, naming both, so a stage fed the wrong stage's output says so instead of misreading it.

A document nests far deeper than the text it came from: every level of a source program costs up to six levels of FON in its syntax tree, so the FON reader's default bound of 64 stops a program about ten levels deep. A document is read with `DEPTH`, a bound past anything the writer can produce in one interpreter: Python's recursion gives out in `write` near fifty source levels, and the in-process chain near ninety, long before the reader would refuse. Its words get the same care: the longest numeral the language admits, `DIGITS` digits, a sign and a point, is one FON word, so `TOKEN` allows exactly that and the reader's default of 4096 would refuse a legal negative.

The representation names are the ones `Pass` carries as its source and target: `Source` (text), `Tokens`, `Syntax` (the surface `Program`), `Core` (the statements), `Stacks` (what evaluation leaves) and `Error`, the `FplError` a stage raised, so a diagnostic can cross the pipe as data.

In [ ]:
@dataclass(frozen=True)
class Codec[A]:
    """
    The FON bytes of one representation, named by `repr`.
    """

    repr: str
    kind: type

    def fonify(self, value: A) -> bytes:
        """
        A → bytes: the value as a document of this representation.
        """
        document = Tagged((Sym(self.repr), to_fon(value)))
        return (write(document) + "\n").encode()

    def defonify(self, data: bytes) -> A:
        """
        bytes → A; a document of another representation, or one whose
        payload is not of this representation's kind, is refused.
        """
        got, payload = _document(data)
        if got != self.repr:
            raise ValueError(f"expected {self.repr}, got {got}")
        value = from_fon(payload)
        if not isinstance(value, self.kind):
            kind = type(value).__name__
            raise ValueError(f"{self.repr} document holds {kind}")
        return cast(A, value)


DEPTH = 1024
"""
The deepest nesting a document may have.
"""

TOKEN = ast_core.DIGITS + 2
"""
The longest word a document may have: a numeral of `DIGITS` digits and
its sign and point.
"""


def _document(data: bytes) -> tuple[str, Value]:
    """
    A document's representation name and value; anything else, FON that
    does not read included, is refused with a ValueError.
    """
    try:
        value = read(data.decode(), max_depth=DEPTH, max_token=TOKEN)
    except FplError as error:
        raise ValueError(f"not FON: {error.message}") from None
    match value:
        case Tagged(items=(Sym(name=name), payload)):
            return name, payload
        case _:
            raise ValueError("not a FON document")


def read_repr(data: bytes) -> str:
    """
    The name of the representation a document carries.
    """
    return _document(data)[0]


SOURCE = Codec[str]("Source", str)
TOKENS = Codec[Tokens]("Tokens", Tokens)
SYNTAX = Codec[ast_surface.Program]("Syntax", ast_surface.Program)
CORE = Codec[tuple[ast_core.Statement, ...]]("Core", tuple)
STACKS = Codec[tuple[tuple[ast_core.Value, ...], ...]]("Stacks", tuple)
ERROR = Codec[FplError]("Error", FplError)
CODECS: dict[str, Codec[object]] = {
    codec.repr: cast(Codec[object], codec)
    for codec in (SOURCE, TOKENS, SYNTAX, CORE, STACKS, ERROR)
}